# Dataset baseline: nvpdyf-bdd100k

Этот ноутбук покрывает следующие задачи:

1. Поиск и проверка датасета;
2. Базовая информация о датасете — количество объектов в разбивках, распределение по классам, проверки качества данных (пропущенные/пустые/«осиротевшие» разметки);
3. Отрисовка нескольких сцен с рамками ground-truth для визуальной проверки перед тем, как на этих данных начнётся обучение.

Вся основная логика находится в [`src/datasets/nvpdyf_bdd100k.py`](src/datasets/nvpdyf_bdd100k.py)
и [`src/utils/visualize.py`](src/utils/visualize.py) - этот нотбук просто вызывает ее и показывает результаты.


In [ ]:
import logging
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from omegaconf import OmegaConf

from src.datasets import nvpdyf_bdd100k as ds
from src.utils.visualize import draw_ground_truth

# проект обычно логгирует через Hydra (см. train.py); здесь мы хотим, чтобы
# logger.info(...) просто печатался в консоль
logging.basicConfig(level=logging.INFO, format="%(message)s")


## 1. Поиск датасета

Указывайте `input_dir` на корень монтирования, а не на конкретную
подпапку датасета — точный путь в Kaggle может отличаться, а поиск ниже
рекурсивный, поэтому он найдёт датасет в любом случае.

- В Kaggle: `/kaggle/input` (значение по умолчанию ниже).
- Локально: переопределите `INPUT_DIR` на папку, куда вы загрузили
  `nvpdyf-bdd100k`, или укажите значение по умолчанию из
  `src/configs/datasets/nvpdyf_bdd100k.yaml`.

In [ ]:
# Значение input_dir по умолчанию, взятое из конфига Hydra, который использует train.py
_cfg = OmegaConf.load("src/configs/datasets/nvpdyf_bdd100k.yaml")

INPUT_DIR = Path("/kaggle/input")  # <- переопределите для локального запуска
if not INPUT_DIR.exists():
    INPUT_DIR = Path(_cfg.input_dir)

DATA_ROOT = ds.find_dataset_root(INPUT_DIR)
CLASSES = ds.load_classes(DATA_ROOT)

print(f"dataset root: {DATA_ROOT}")
print(f"classes ({len(CLASSES)}): {CLASSES}")


## 2. Информация о датасете

Количество кадров и рамок по разбивкам, а также несколько проверок качества,
о которых полезно знать *до* начала обучения:

- **missing_labels** — изображения без соответствующего `.txt`-файла
  (Ultralytics считает их неразмеченными, а не пустыми — это отличается от
  намеренного кадра «без объектов»);
- **empty_labels** — `.txt`-файлы с нулевым количеством строк (кадр без объектов,
  например, пустая дорога — допустимо, но полезно знать количество);
- **orphan_labels** — `.txt`-файлы без соответствующего изображения (мёртвый груз,
  обычно признак проблемы упаковки датасета).

In [ ]:
SPLITS = [s for s in ("train", "val", "test") if (DATA_ROOT / "images" / s).is_dir()]

stats = {split: ds.collect_stats(DATA_ROOT, split) for split in SPLITS}
ds.describe_dataset(stats, CLASSES)


### Распределение по классам

Один столбец на класс, суммированный по всем разбивкам, отсортированный по убыванию.

In [ ]:
total_counts = Counter()
for s in stats.values():
    total_counts.update(s["class_counts"])

# Включаем каждый класс из data.yaml, даже если у него 0 рамок — класс,
# полностью отсутствующий в датасете, легко не заметить, если он просто пропадёт
# из графика, а столбец нулевой длины для него будет заметен.
items = sorted(
    ((cid, total_counts[cid]) for cid in CLASSES), key=lambda kv: kv[1]
)  # ascending, for barh
labels = [CLASSES[cid] for cid, _ in items]
values = [v for _, v in items]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(labels, values, color="#4C72B0")
ax.set_xlabel("boxes (all splits)")
ax.set_title("Class distribution - nvpdyf-bdd100k")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="0.9", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for bar, v in zip(bars, values):
    ax.text(bar.get_width(), bar.get_y() + bar.get_height() / 2, f" {v}", va="center", fontsize=9)
plt.tight_layout()
plt.show()


## 3. Несколько сцен

Два случайных кадра (один из `train`, один из `val`) с нанесёнными
рамками ground-truth — быстрая проверка того, что изображения и разметка
действительно соответствуют друг другу, перед тем как на этих данных
начнётся обучение. При повторном запуске с тем же `seed` результат детерминирован.

In [ ]:
SEED = _cfg.sample_seed

scenes = []
for split in ("train", "val"):
    if split in SPLITS:
        scenes += ds.sample_scenes(DATA_ROOT, split, n=1, seed=SEED)

fig = draw_ground_truth(scenes, CLASSES)
